# CS771 Minor 1: LinearSVC Hyperparameter Optimization

This notebook demonstrates the effect of regularization (C), convergence tolerance (	ol), and maximum iterations (max_iter) on the Linear Support Vector Classifier (LinearSVC) for high-dimensional, linearly separable datasets.

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC
import genSyntheticData as gsd
import plotData as pd
from matplotlib import pyplot as plt
import time
import warnings
import json
warnings.filterwarnings("ignore")

### Helpers

In [ ]:
def train_eval(X, y, C=1.0, max_iter=1000, tol=1e-4,
               penalty='l2', loss='squared_hinge', n_runs=5):
    clf = LinearSVC(penalty=penalty, loss=loss, C=C,
                    tol=tol, max_iter=int(max_iter), dual='auto')
    t_total = 0.0
    for _ in range(n_runs):
        tic = time.perf_counter()
        clf.fit(X, y)
        toc = time.perf_counter()
        t_total += toc - tic
    return np.average(y == clf.predict(X)), t_total / n_runs, clf

def plot_boundary(clf, XPos, XNeg, level, title, filename):
    w = clf.coef_.reshape(-1)
    b = clf.intercept_[0]
    mySVM = lambda X: X.dot(w) + b
    X_all = np.vstack((XPos, XNeg))
    xl = np.max(np.abs(X_all[:, 0])) * 1.1
    yl = np.max(np.abs(X_all[:, 1])) * 1.1
    fig = pd.getFigure(7, 7)
    pd.shade2D(mySVM, fig, mode='batch', xlim=xl, ylim=yl)
    pd.plot2D(XPos, fig, color='m', marker='+', label='Positive')
    pd.plot2D(XNeg, fig, color='y', marker='o', label='Negative')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  -> Saved {filename}")

def sweep(X, y, param_name, values, defaults, n_runs=5):
    """Sweep one hyperparameter. Returns list of (value, acc, time_ms)."""
    results = []
    for v in values:
        kwargs = dict(defaults)
        kwargs[param_name] = v
        acc, t, clf = train_eval(X, y, n_runs=n_runs, **kwargs)
        results.append((v, acc, t * 1000))
    return results

def print_sweep(results, param_label):
    print(f"  {'Value':>12}  {'Acc':>8}  {'Time(ms)':>10}")
    print(f"  {'-'*36}")
    for v, acc, t in results:
        tag = " ***" if acc >= 1.0 - 1e-9 else ""
        print(f"  {v:>12g}  {acc:>8.2%}  {t:>10.4f}{tag}")

### Q2 - Small Challenge (level=0.3, n=500)

In [ ]:
print("=" * 70)
print("Q2: SMALL CHALLENGE - Fine-grained search (level=0.3, n=500)")
print("=" * 70)

myLevel_q2 = 0.3
n_q2 = 500
XPos, yPos, XNeg, yNeg = gsd.genChallengeData(level=myLevel_q2, n=n_q2)
X_q2 = np.vstack((XPos, XNeg))
y_q2 = np.concatenate((yPos, yNeg))

defaults_q2 = dict(C=1.0, max_iter=1000, tol=1e-4)

print("\n--- Phase 1: Coarse C search (others at defaults) ---")
C_coarse = [1e-5, 1e-4, 1e-3, 5e-3, 0.01, 0.05, 0.1, 1, 10, 100, 1000]
res_C_q2 = sweep(X_q2, y_q2, 'C', C_coarse, defaults_q2)
print_sweep(res_C_q2, 'C')

best_C_q2 = None
for v, acc, _ in res_C_q2:
    if acc >= 1.0 - 1e-9:
        best_C_q2 = v
        break

if best_C_q2 is not None and best_C_q2 > C_coarse[0]:
    idx = C_coarse.index(best_C_q2)
    lo = C_coarse[idx - 1] if idx > 0 else best_C_q2 / 10
    hi = best_C_q2
    print(f"\n--- Phase 2: Fine C search between {lo} and {hi} ---")
    if hi <= 1:
        fine_vals = sorted(set(np.round(np.arange(lo, hi + lo/2, (hi-lo)/10), 6)))
    else:
        fine_vals = sorted(set(np.round(np.linspace(lo, hi, 20), 4)))
    res_C_q2_fine = sweep(X_q2, y_q2, 'C', fine_vals, defaults_q2)
    print_sweep(res_C_q2_fine, 'C')
    for v, acc, _ in res_C_q2_fine:
        if acc >= 1.0 - 1e-9:
            best_C_q2 = v
            break
    res_C_q2 = res_C_q2 + res_C_q2_fine

print(f"\n  => Smallest C for 100%: {best_C_q2}")

print("\n--- Phase 1: Coarse max_iter search ---")
mi_coarse = [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000, 5000]
res_mi_q2 = sweep(X_q2, y_q2, 'max_iter', mi_coarse, defaults_q2)
print_sweep(res_mi_q2, 'max_iter')

best_mi_q2 = None
for v, acc, _ in res_mi_q2:
    if acc >= 1.0 - 1e-9:
        best_mi_q2 = int(v)
        break

if best_mi_q2 is not None and best_mi_q2 > 1:
    idx = mi_coarse.index(best_mi_q2)
    lo = mi_coarse[idx - 1] if idx > 0 else 1
    hi = best_mi_q2
    print(f"\n--- Phase 2: Fine max_iter search between {lo} and {hi} ---")
    fine_vals = list(range(lo, hi + 1))
    res_mi_q2_fine = sweep(X_q2, y_q2, 'max_iter', fine_vals, defaults_q2)
    print_sweep(res_mi_q2_fine, 'max_iter')
    for v, acc, _ in res_mi_q2_fine:
        if acc >= 1.0 - 1e-9:
            best_mi_q2 = int(v)
            break
    res_mi_q2 = res_mi_q2 + res_mi_q2_fine

print(f"\n  => Smallest max_iter for 100%: {best_mi_q2}")

print("\n--- Phase 1: Coarse tol search (large -> small) ---")
tol_coarse = [50, 10, 5, 1, 0.5, 0.1, 0.01, 0.001, 1e-4, 1e-5, 1e-6]
res_tol_q2 = sweep(X_q2, y_q2, 'tol', tol_coarse, defaults_q2)
print_sweep(res_tol_q2, 'tol')

best_tol_q2 = None
for v, acc, _ in sorted(res_tol_q2, key=lambda x: -x[0]):
    if acc >= 1.0 - 1e-9:
        best_tol_q2 = v
        break

if best_tol_q2 is not None:
    sorted_tols = sorted(tol_coarse)
    idx = sorted_tols.index(best_tol_q2)
    if idx < len(sorted_tols) - 1:
        lo = best_tol_q2
        hi = sorted_tols[idx + 1]
        print(f"\n--- Phase 2: Fine tol search between {lo} and {hi} ---")
        fine_vals = sorted(set(np.round(np.linspace(lo, hi, 20), 6)))
        res_tol_q2_fine = sweep(X_q2, y_q2, 'tol', fine_vals, defaults_q2)
        print_sweep(res_tol_q2_fine, 'tol')
        for v, acc, _ in sorted(res_tol_q2_fine, key=lambda x: -x[0]):
            if acc >= 1.0 - 1e-9:
                best_tol_q2 = v
                break
        res_tol_q2 = res_tol_q2 + res_tol_q2_fine

print(f"\n  => Largest tol for 100%: {best_tol_q2}")

res_C_q2_all = sorted(set((v, acc, t) for v, acc, t in res_C_q2), key=lambda x: x[0])
res_mi_q2_all = sorted(set((v, acc, t) for v, acc, t in res_mi_q2), key=lambda x: x[0])
res_tol_q2_all = sorted(set((v, acc, t) for v, acc, t in res_tol_q2), key=lambda x: x[0])

print(f"\n  Optimal: C={best_C_q2}, max_iter={best_mi_q2}, tol={best_tol_q2}")
_, _, clf_q2 = train_eval(X_q2, y_q2, C=best_C_q2, max_iter=best_mi_q2, tol=best_tol_q2)
plot_boundary(clf_q2, XPos, XNeg, myLevel_q2,
              f"Q2: level={myLevel_q2}, C={best_C_q2}, max_iter={best_mi_q2}, tol={best_tol_q2}",
              "v2_q2_boundary.png")

### Q3 - Large Challenge (level=55, n=500)

In [ ]:
print("\n" + "=" * 70)
print("Q3: LARGE CHALLENGE - Fine-grained search (level=55, n=500)")
print("=" * 70)

myLevel_q3 = 55
n_q3 = 500
XPos, yPos, XNeg, yNeg = gsd.genChallengeData(level=myLevel_q3, n=n_q3)
X_q3 = np.vstack((XPos, XNeg))
y_q3 = np.concatenate((yPos, yNeg))

defaults_q3 = dict(C=100.0, max_iter=1000, tol=1e-4)

print("\n--- Phase 1: Coarse C search ---")
C_coarse_q3 = [1e-5, 1e-4, 1e-3, 5e-3, 0.01, 0.05, 0.1, 1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1000]
res_C_q3 = sweep(X_q3, y_q3, 'C', C_coarse_q3, defaults_q3)
print_sweep(res_C_q3, 'C')

best_C_q3 = None
for v, acc, _ in res_C_q3:
    if acc >= 1.0 - 1e-9:
        best_C_q3 = v
        break

if best_C_q3 is not None and best_C_q3 > C_coarse_q3[0]:
    idx = [i for i, x in enumerate(C_coarse_q3) if x == best_C_q3][0]
    lo = C_coarse_q3[idx - 1] if idx > 0 else best_C_q3 / 10
    hi = best_C_q3
    print(f"\n--- Phase 2: Fine C search between {lo} and {hi} ---")
    step = max(1, (hi - lo) / 20)
    fine_vals = sorted(set(np.round(np.arange(lo, hi + 0.5, step), 2)))
    res_C_q3_fine = sweep(X_q3, y_q3, 'C', fine_vals, defaults_q3)
    print_sweep(res_C_q3_fine, 'C')
    for v, acc, _ in res_C_q3_fine:
        if acc >= 1.0 - 1e-9:
            best_C_q3 = v
            break
    res_C_q3 = res_C_q3 + res_C_q3_fine

print(f"\n  => Smallest C for 100%: {best_C_q3}")

print("\n--- Phase 1: Coarse max_iter search ---")
mi_coarse_q3 = [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 50, 100, 200, 500, 1000, 5000]
res_mi_q3 = sweep(X_q3, y_q3, 'max_iter', mi_coarse_q3, dict(C=best_C_q3 if best_C_q3 else 100, max_iter=1000, tol=1e-4))
print_sweep(res_mi_q3, 'max_iter')

best_mi_q3 = None
for v, acc, _ in res_mi_q3:
    if acc >= 1.0 - 1e-9:
        best_mi_q3 = int(v)
        break

if best_mi_q3 is not None and best_mi_q3 > 1:
    idx = [i for i, x in enumerate(mi_coarse_q3) if x == best_mi_q3][0]
    lo = mi_coarse_q3[idx - 1] if idx > 0 else 1
    hi = best_mi_q3
    if hi - lo > 1:
        print(f"\n--- Phase 2: Fine max_iter search between {lo} and {hi} ---")
        fine_vals = list(range(lo, hi + 1))
        res_mi_q3_fine = sweep(X_q3, y_q3, 'max_iter', fine_vals,
                               dict(C=best_C_q3 if best_C_q3 else 100, max_iter=1000, tol=1e-4))
        print_sweep(res_mi_q3_fine, 'max_iter')
        for v, acc, _ in res_mi_q3_fine:
            if acc >= 1.0 - 1e-9:
                best_mi_q3 = int(v)
                break
        res_mi_q3 = res_mi_q3 + res_mi_q3_fine

print(f"\n  => Smallest max_iter for 100%: {best_mi_q3}")

print("\n--- Phase 1: Coarse tol search ---")
tol_coarse_q3 = [1e-6, 1e-5, 1e-4, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009,
                  0.01, 0.1, 0.5, 1, 5, 10, 50]
res_tol_q3 = sweep(X_q3, y_q3, 'tol', tol_coarse_q3,
                    dict(C=best_C_q3 if best_C_q3 else 100, max_iter=best_mi_q3 if best_mi_q3 else 1000, tol=1e-4))
print_sweep(res_tol_q3, 'tol')

best_tol_q3 = None
for v, acc, _ in sorted(res_tol_q3, key=lambda x: -x[0]):
    if acc >= 1.0 - 1e-9:
        best_tol_q3 = v
        break

if best_tol_q3 is not None:
    sorted_tols_q3 = sorted(tol_coarse_q3)
    idx = sorted_tols_q3.index(best_tol_q3)
    if idx < len(sorted_tols_q3) - 1:
        lo = best_tol_q3
        hi = sorted_tols_q3[idx + 1]
        if hi - lo > 1e-7:
            print(f"\n--- Phase 2: Fine tol search between {lo} and {hi} ---")
            fine_vals = sorted(set(np.round(np.linspace(lo, hi, 15), 7)))
            res_tol_q3_fine = sweep(X_q3, y_q3, 'tol', fine_vals,
                                    dict(C=best_C_q3, max_iter=best_mi_q3, tol=1e-4))
            print_sweep(res_tol_q3_fine, 'tol')
            for v, acc, _ in sorted(res_tol_q3_fine, key=lambda x: -x[0]):
                if acc >= 1.0 - 1e-9:
                    best_tol_q3 = v
                    break
            res_tol_q3 = res_tol_q3 + res_tol_q3_fine

print(f"\n  => Largest tol for 100%: {best_tol_q3}")

res_C_q3_all = sorted(set((v, acc, t) for v, acc, t in res_C_q3), key=lambda x: x[0])
res_mi_q3_all = sorted(set((v, acc, t) for v, acc, t in res_mi_q3), key=lambda x: x[0])
res_tol_q3_all = sorted(set((v, acc, t) for v, acc, t in res_tol_q3), key=lambda x: x[0])

print(f"\n  Optimal: C={best_C_q3}, max_iter={best_mi_q3}, tol={best_tol_q3}")
_, _, clf_q3 = train_eval(X_q3, y_q3, C=best_C_q3 or 100, max_iter=best_mi_q3 or 1000, tol=best_tol_q3 or 1e-4)
plot_boundary(clf_q3, XPos, XNeg, myLevel_q3,
              f"Q3: level={myLevel_q3}, C={best_C_q3}, max_iter={best_mi_q3}, tol={best_tol_q3}",
              "v2_q3_boundary.png")

### Q4 - Medium Challenge (level=5, n=100) - Fine-grained sweeps

In [ ]:
print("\n" + "=" * 70)
print("Q4: MEDIUM CHALLENGE - Detailed hyperparameter effects (level=5, n=100)")
print("=" * 70)

myLevel_q4 = 5
n_q4 = 100
XPos, yPos, XNeg, yNeg = gsd.genChallengeData(level=myLevel_q4, n=n_q4)
X_q4 = np.vstack((XPos, XNeg))
y_q4 = np.concatenate((yPos, yNeg))

print("\n--- Effect of C (max_iter=1000, tol=1e-4) ---")
C_vals_q4 = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100, 500, 1000]
res_C_q4 = sweep(X_q4, y_q4, 'C', C_vals_q4, dict(C=1, max_iter=1000, tol=1e-4))
print_sweep(res_C_q4, 'C')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_C_q4], [r[1]*100 for r in res_C_q4], 'bo-', label='Accuracy (%)')
ax.set_xscale('log'); ax.set_xlabel('C'); ax.set_ylabel('Training Accuracy (%)')
ax.set_title('Q4: Effect of C on Accuracy (level=5, n=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q4_C_accuracy.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_C_accuracy.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_C_q4], [r[2] for r in res_C_q4], 'rs-', label='Train Time (ms)')
ax.set_xscale('log'); ax.set_xlabel('C'); ax.set_ylabel('Training Time (ms)')
ax.set_title('Q4: Effect of C on Training Time (level=5, n=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q4_C_time.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_C_time.png")

print("\n--- Effect of max_iter (C=1, tol=1e-4) ---")
mi_vals_q4 = [1, 2, 3, 5, 7, 10, 15, 20, 50, 100, 200, 500, 1000, 5000, 10000]
res_mi_q4 = sweep(X_q4, y_q4, 'max_iter', mi_vals_q4, dict(C=1, max_iter=1000, tol=1e-4))
print_sweep(res_mi_q4, 'max_iter')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_mi_q4], [r[1]*100 for r in res_mi_q4], 'bo-', label='Accuracy (%)')
ax.set_xscale('log'); ax.set_xlabel('max_iter'); ax.set_ylabel('Training Accuracy (%)')
ax.set_title('Q4: Effect of max_iter on Accuracy (level=5, n=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q4_maxiter_accuracy.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_maxiter_accuracy.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_mi_q4], [r[2] for r in res_mi_q4], 'rs-', label='Train Time (ms)')
ax.set_xscale('log'); ax.set_xlabel('max_iter'); ax.set_ylabel('Training Time (ms)')
ax.set_title('Q4: Effect of max_iter on Training Time (level=5, n=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q4_maxiter_time.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_maxiter_time.png")

print("\n--- Effect of tol (C=1, max_iter=1000) ---")
tol_vals_q4 = [1e-6, 1e-5, 1e-4, 5e-4, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50]
res_tol_q4 = sweep(X_q4, y_q4, 'tol', tol_vals_q4, dict(C=1, max_iter=1000, tol=1e-4))
print_sweep(res_tol_q4, 'tol')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_tol_q4], [r[1]*100 for r in res_tol_q4], 'bo-', label='Accuracy (%)')
ax.set_xscale('log'); ax.set_xlabel('tol'); ax.set_ylabel('Training Accuracy (%)')
ax.set_title('Q4: Effect of tol on Accuracy (level=5, n=100)'); ax.legend()
ax.invert_xaxis()
plt.tight_layout(); plt.savefig('v2_q4_tol_accuracy.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_tol_accuracy.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_tol_q4], [r[2] for r in res_tol_q4], 'rs-', label='Train Time (ms)')
ax.set_xscale('log'); ax.set_xlabel('tol'); ax.set_ylabel('Training Time (ms)')
ax.set_title('Q4: Effect of tol on Training Time (level=5, n=100)'); ax.legend()
ax.invert_xaxis()
plt.tight_layout(); plt.savefig('v2_q4_tol_time.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q4_tol_time.png")

### Q5 - Effect of Sample Size (level=100)

In [ ]:
print("\n" + "=" * 70)
print("Q5: SAMPLE SIZE EFFECT (level=100)")
print("=" * 70)

myLevel_q5 = 100
n_values_q5 = [10, 25, 50, 75, 100, 200, 500, 750, 1000]
res_q5 = []

for nv in n_values_q5:
    XPos, yPos, XNeg, yNeg = gsd.genChallengeData(level=myLevel_q5, n=nv)
    X = np.vstack((XPos, XNeg))
    y = np.concatenate((yPos, yNeg))
    acc, t, _ = train_eval(X, y, C=1, max_iter=1000, tol=1e-4)
    res_q5.append((nv, acc, t * 1000))
    print(f"  n={nv:>5d}  acc={acc:.4f}  time={t*1000:.3f}ms")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_q5], [r[1]*100 for r in res_q5], 'bo-', label='Accuracy (%)')
ax.set_xscale('log'); ax.set_xlabel('n (points per cluster)'); ax.set_ylabel('Training Accuracy (%)')
ax.set_title('Q5: Effect of Sample Size on Accuracy (level=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q5_accuracy.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q5_accuracy.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in res_q5], [r[2] for r in res_q5], 'rs-', label='Train Time (ms)')
ax.set_xscale('log'); ax.set_xlabel('n (points per cluster)'); ax.set_ylabel('Training Time (ms)')
ax.set_title('Q5: Effect of Sample Size on Training Time (level=100)'); ax.legend()
plt.tight_layout(); plt.savefig('v2_q5_time.png', dpi=150, bbox_inches='tight'); plt.close()
print("  -> Saved v2_q5_time.png")

### Save all results to JSON for LaTeX table generation

In [ ]:
all_results = {
    'q2': {
        'level': myLevel_q2, 'n': n_q2,
        'best_C': best_C_q2, 'best_mi': best_mi_q2, 'best_tol': best_tol_q2,
        'C_results': [(float(v), float(a), float(t)) for v, a, t in res_C_q2_all],
        'mi_results': [(float(v), float(a), float(t)) for v, a, t in res_mi_q2_all],
        'tol_results': [(float(v), float(a), float(t)) for v, a, t in res_tol_q2_all],
    },
    'q3': {
        'level': myLevel_q3, 'n': n_q3,
        'best_C': float(best_C_q3) if best_C_q3 else None,
        'best_mi': int(best_mi_q3) if best_mi_q3 else None,
        'best_tol': float(best_tol_q3) if best_tol_q3 else None,
        'C_results': [(float(v), float(a), float(t)) for v, a, t in res_C_q3_all],
        'mi_results': [(float(v), float(a), float(t)) for v, a, t in res_mi_q3_all],
        'tol_results': [(float(v), float(a), float(t)) for v, a, t in res_tol_q3_all],
    },
    'q4': {
        'level': myLevel_q4, 'n': n_q4,
        'C_results': [(float(v), float(a), float(t)) for v, a, t in res_C_q4],
        'mi_results': [(float(v), float(a), float(t)) for v, a, t in res_mi_q4],
        'tol_results': [(float(v), float(a), float(t)) for v, a, t in res_tol_q4],
    },
    'q5': {
        'level': myLevel_q5,
        'results': [(int(v), float(a), float(t)) for v, a, t in res_q5],
    },
}

with open('v2_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("\n  -> Saved v2_results.json")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE!")
print("=" * 70)